In [ ]:
# run this notebook after 15_prepare_to_calculate_af_R
# use this notebook calculate allele frequencies for trio and sibling differences after removing first-degree relatives
# after this, run 17_determine_af_and_cluster_filter_using_quads_R
# (note that running this analysis was very difficult)

In [ ]:
%%bash 

gsutil cp ./all_snps_QC_for_AF.txt $WORKSPACE_BUCKET/data
gsutil cp ./samples_to_remove.txt $WORKSPACE_BUCKET/data

In [ ]:
import hail as hl
import os, ast, re
import pandas as pd

bucket=os.getenv('WORKSPACE_BUCKET')
hl.init()

In [ ]:
difs_table = pd.read_csv(f"{bucket}/data/all_snps_QC_for_AF.txt", sep="\t")
difs_table['vid'] = (
    difs_table['locus'].str.replace(r'^chr', '', regex=True).str.replace(':', '-', n=1)
    + '-' +
    difs_table['alleles'].apply(lambda s: '-'.join(ast.literal_eval(s)))
)
vids = list(set(difs_table['vid'].tolist()))
vids_table = hl.Table.from_pandas(pd.DataFrame({'vid': vids}), key=['vid'])

auxiliary_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux"
vat_path = f'{auxiliary_path}/vat/vat_complete.bgz.tsv.gz'
vat_table = hl.import_table(vat_path, force=True, quote='"', delimiter="\t", force_bgz=True)
vat_new = (vat_table
    .select('vid','contig','position','genomic_location','dbsnp_rsid','ref_allele',
            'alt_allele','gvs_all_ac','gvs_all_an','gvs_all_af')
    .key_by('vid').distinct().semi_join(vids_table).repartition(50))
vat_new.export(f'{bucket}/data/all_snps_AF_v8.tsv')

In [ ]:
samples = pd.read_csv(f'{bucket}/data/samples_to_remove.txt', sep="\t", header=None)
samples = samples.iloc[:, 0].astype(str).tolist()

sites = hl.import_table(f"{bucket}/data/all_snps_QC_for_AF.txt", impute=True)
parts = sites.locus.split(":")
sites = sites.annotate(
    locus = hl.locus(parts[0], hl.int(parts[1]), reference_genome='GRCh38')
).key_by('locus')

vds = hl.vds.read_vds(os.getenv("WGS_VDS_PATH"))
vds = hl.vds.filter_samples(vds, samples, keep=True)        # filters ref + variant together

vd = vds.variant_data
vd = vd.filter_rows(hl.is_defined(sites[vd.locus]))         # restrict rows before densify
vds = hl.vds.VariantDataset(vds.reference_data, vd)

mt = hl.vds.to_dense_mt(vds)                                # ref samples -> lgt 0/0
mt = mt.annotate_entries(GT = hl.vds.lgt_to_gt(mt.LGT, mt.LA))   # after densify
mt = mt.annotate_entries(
    GT = hl.if_else(hl.coalesce(mt.FT, True), mt.GT, hl.missing(hl.tcall))
)
mt = mt.annotate_rows(cs = hl.agg.call_stats(mt.GT, mt.alleles))
mt = mt.checkpoint(f'{bucket}/data/sib_trio_VAT_GIAB_samples_to_remove.mt', overwrite=True)
mt.rows().select('cs').flatten().export(f"{bucket}/data/samples_to_remove_AC_AN.tsv")


In [ ]:
rem_raw = pd.read_csv(f"{bucket}/data/samples_to_remove_AC_AN.tsv", sep="\t")

def explode_rem(row):
    alleles = ast.literal_eval(row['alleles'])
    ac      = ast.literal_eval(str(row['cs.AC']))
    an      = row['cs.AN']
    cp      = re.sub(r'^chr', '', str(row['locus'])).replace(':', '-', 1)
    ref     = alleles[0]
    return [{'vid': f'{cp}-{ref}-{alleles[i]}', 'AC': ac[i], 'AN': an}
            for i in range(1, len(alleles))]

rem = pd.DataFrame([r for _, row in rem_raw.iterrows() for r in explode_rem(row)])
rem['AN'] = pd.to_numeric(rem['AN'], errors='coerce')
rem = rem.drop_duplicates('vid')

af = pd.read_csv(f"{bucket}/data/all_snps_AF_v8.tsv", sep="\t").drop_duplicates('vid')
for c in ['gvs_all_ac', 'gvs_all_an', 'gvs_all_af']:
    af[c] = pd.to_numeric(af[c], errors='coerce')

af = af.merge(rem, on='vid', how='left')
af['AC'] = af['AC'].fillna(0)
af['AN'] = af['AN'].fillna(0)
af['new_ac'] = af['gvs_all_ac'] - af['AC']
af['new_an'] = af['gvs_all_an'] - af['AN']
af['new_af'] = af['new_ac'] / af['new_an'].where(af['new_an'] > 0)
af.to_csv(f"{bucket}/data/all_snps_AF_v8_new_AF.tsv", sep="\t", index=False)

master = difs_table.merge(af[['vid','new_ac','new_an','new_af']], on='vid', how='left')
master.to_csv(f"{bucket}/data/all_snps_QC_for_AF_new_AF.txt", sep="\t", index=False)

In [ ]:
%%bash 

gsutil cp ./all_snps_QC_for_AF_new_AF.txt .